# ESM3 Multimodal Protein Language Model

This notebook demonstrates the three core operations of the ESM3 tool: extracting sequence embeddings, sampling mutations, and scoring sequences. Unlike sequence-only models, ESM3 was trained on multimodal protein data covering sequence, structure, and function, allowing it to capture richer relationships between amino acid identity, three-dimensional geometry, and biological annotation. Even when working with sequence inputs alone, ESM3's representations encode structural and functional context that purely sequence-based models miss.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("esm3")
display_overview("esm3")
display_docs_section("esm3", "Background")

# ESM3

> ESM3 is EvolutionaryScale's next-generation [protein language model](https://www.evolutionaryscale.ai/blog/esm-cambrian) with sequence and structure modeling capabilities. This package's `esm3-sample` tool exposes masked sequence editing over supplied protein sequences. The open model (`esm3_sm_open_v1`) provides embeddings, logits, masked sampling, and scoring in a unified framework.

## Background

**What are protein language models?**
Protein language models (pLMs) learn the "grammar" of proteins from evolutionary data. ESM3 extends this by jointly modeling sequence and structure, capturing:

* **Sequence conservation**: Which residues are essential for function
* **[Co-evolution](https://en.wikipedia.org/wiki/Coevolution)**: Pairs of residues that evolve together (often in contact)
* **Structural constraints**: Patterns that define [secondary/tertiary structure](https://en.wikipedia.org/wiki/Protein_structure)
* **3D geometry**: Spatial relationships between residues

**Why ESM3 over ESM2?**
ESM3 is broader than ESM2 at the model-family level. In Proto Tools today, use `esm3-sample` for masked sequence editing and local refinement of supplied protein sequences; for pure sequence embedding tasks, ESM2 is often faster.

## Available tools

In [2]:
display_available_tools("esm3")

- **`run_esm3_embeddings()`** — Extract protein sequence embeddings and logits using ESM3
- **`run_esm3_sample()`** — Sample protein sequences using ESM3 language model
- **`run_esm3_score()`** — Score protein sequences using ESM3 language model

### `run_esm3_embeddings`

ESM3 produces dense vector representations for each input protein sequence. Because the model was trained on multimodal data, these embeddings encode not only evolutionary patterns but also structural and functional context. The mean-pooled vectors can be used for similarity search, clustering, classification, and as input features for downstream machine learning models.

In [3]:
from proto_tools import (
    ESM3EmbeddingsConfig,
    ESM3EmbeddingsInput,
    run_esm3_embeddings,
)

In [4]:
# Display docs
display_api_reference("esm3", "input", "run_esm3_embeddings")

# Two hemoglobin-like sequences to embed
sequences = [
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
    "MNLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG",
]
inputs = ESM3EmbeddingsInput(sequences=sequences)

**Input** — `MaskedModelInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Protein sequence(s) to process. Can be provided as: |

In [5]:
# Display config docs
display_api_reference("esm3", "config", "run_esm3_embeddings")

# Configure the open-source ESM3 model | see docs above for all fields
config = ESM3EmbeddingsConfig(
    model_checkpoint="esm3_sm_open_v1",
    batch_size=2,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESM3EmbeddingsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `string` | `esm3_sm_open_v1` | ESM3 model checkpoint to use. Currently available: |
| `return_logits` | `boolean` | `False` | Whether to include per-position logits in the output. When `True`, returns logits for each sequence. When `False`, only returns metrics (saves memory and serialization time). Default: `False`. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `batch_size` | `integer` | `1` | Number of sequences to process in parallel. Larger batches improve throughput but require more GPU memory. Optimal values depend on GPU memory, model size, and sequence lengths. Typical values range from 1 (safest) to 128 (faster, more memory). Default: 1. |
| `verbose` | `boolean` | `False` | Whether to print status messages during model execution, including loading progress and timing information. Default: `False`. |
| `device` | `string` | `cuda` | Device to run the model on. Options include `"cuda"` (NVIDIA GPU), `"cpu"` (CPU execution), `"mps"` (Apple Metal), or specific GPU devices like `"cuda:0"`. Default: `"cuda"`. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the embeddings function
embeddings_result = run_esm3_embeddings(inputs, config)

Running run_esm3_embeddings [00:00]

In [7]:
# Display output docs
display_api_reference("esm3", "output", "run_esm3_embeddings")

# Inspect the embedding dimension and first few values for each sequence
for i, emb in enumerate(embeddings_result.results):
    print(f"Sequence {i + 1}: embedding length = {len(emb.mean_embedding)}, first 5 values = {emb.mean_embedding[:5]}")

**Output** — `ESM3EmbeddingsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `List[SequenceEmbedding]` | required | Per-sequence embedding results. Each `SequenceEmbedding` contains: |
| `mean_embedding` | `List[number]` | required | Mean-pooled embedding vector for one sequence. |
| `attention_mask` | `List[integer]` | required | Binary mask indicating valid positions (1) vs padding (0). |
| `logits` | `array` |  | Optional per-position amino acid logits for one sequence. |

Sequence 1: embedding length = 1536, first 5 values = [-20.375, 30.75, 16.125, -73.0, 111.0]
Sequence 2: embedding length = 1536, first 5 values = [-26.75, 22.0, 17.25, -75.0, 105.0]


### `run_esm3_sample`

ESM3 sampling uses the model's predicted amino acid distributions to propose mutations at masked positions. Because ESM3 was trained on multimodal data, its position confidence estimates reflect structural and functional plausibility in addition to evolutionary conservation. There are two ways to specify which positions to sample:

1. **Custom masks** — directly mark positions with `_` in the input sequence to control exactly which residues are redesigned
2. **Masking strategy** — let the `MaskingStrategy` config automatically select positions using random, entropy-based, or max-logit methods

#### Approach 1: Custom masks with `_`

Use the `_` character to explicitly mark which positions should be redesigned. All other positions are held fixed. This is useful when you know exactly which residues you want to mutate — for example, redesigning a loop region or a specific binding interface.

In [8]:
from proto_tools import (
    ESM3SampleConfig,
    ESM3SampleInput,
    run_esm3_sample,
)

# Display input docs
display_api_reference("esm3", "input", "run_esm3_sample")

# Mask positions 6-10 with _ to redesign that region
masked_input = ESM3SampleInput(sequences=["MVLSP_____VKAAW"])

**Input** — `MaskedModelInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Protein sequence(s) to process. Can be provided as: |

In [9]:
# Display config docs
display_api_reference("esm3", "config", "run_esm3_sample")

# No masking strategy needed — positions are already specified by _
config = ESM3SampleConfig(
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESM3SampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `masking_strategy` | `MaskingStrategy` |  | Controls which positions to mask for sampling. Default: random 30%. |
| `method` | `enum` | `random` | Scoring method for position selection. `"random"`: uniform random, `"entropy"`: highest model uncertainty, `"max-logit"`: lowest model confidence. Available options: `random`, `entropy`, `max-logit` |
| `num_mutations` | `integer` |  | Exact number of positions to mask per sequence. |
| `mask_fraction` | `number` |  | Fraction of designable positions to mask (e.g. 0.15 for \~15%). |
| `fixed_positions` | `array` |  | 1-indexed positions that must NOT be masked. Applied uniformly to all sequences. |
| `temperature` | `number` | `1.0` | Temperature for position selection. \< 1.0 is greedy, 1.0 uses scores as-is, > 1.0 is more uniform. |
| `model_name` | `string` |  | Which masked model to use for scoring. Defaults to the sampling tool's model when unset. |
| `model_checkpoint` | `string` | `esm3_sm_open_v1` | ESM3 model checkpoint. Default: `"esm3_sm_open_v1"`. |
| `temperature` | `number` | `1.0` | Sampling temperature (\< 1.0 conservative, > 1.0 diverse). Default: 1.0. |
| `batch_size` | `integer` | `1` | Sequences per GPU forward pass. Default: 1. |
| `return_logits` | `boolean` | `False` | Whether to include per-position logits. Default: `False`. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device to run on. Default: `"cuda"`. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [10]:
# Run sampling on the custom-masked input
custom_mask_result = run_esm3_sample(masked_input, config)

Running run_esm3_sample [00:00]

In [11]:
# Display output docs
display_api_reference("esm3", "output", "run_esm3_sample")

# Show the original vs sampled — only the masked positions change
original = "MVLSPADKTNVKAAW"
for i, seq in enumerate(custom_mask_result.sequences):
    diffs = [j + 1 for j, (a, b) in enumerate(zip(original, seq)) if a != b]
    print(f"Original: {original}")
    print(f"Sampled:  {seq}  (mutated positions: {diffs})")

**Output** — `ESM3SampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Sampled or mutated protein sequences. Each sequence is a string of amino acid characters and is a modified version of the input sequence with masked positions changed to model-predicted alternatives. |
| `logits` | `array` |  | Per-position logits for each sequence. Shape is (num\_sequences, seq\_len, vocab\_size=20). Only present if return\_logits=True in config. |

Original: MVLSPADKTNVKAAW
Sampled:  MVLSPRSLGSVKAAW  (mutated positions: [6, 7, 8, 9, 10])


#### Approach 2: Automatic position selection with `MaskingStrategy`

Instead of manually placing `_` characters, you can let `MaskingStrategy` choose which positions to mask. The `method` parameter controls the selection logic:

- **`"random"`** (default) — uniform random selection
- **`"entropy"`** — targets positions where ESM3 is least certain about the original residue
- **`"max-logit"`** — targets positions where ESM3 most confidently predicts a different amino acid

Use `num_mutations` for an exact count or `mask_fraction` for a proportion of positions.

In [ ]:
from proto_tools.transforms.masking import MaskingStrategy

# Entropy-based masking to target the 5 most uncertain positions
strategy_input = ESM3SampleInput(sequences=["MVLSPADKTNVKAAW"])
strategy_config = ESM3SampleConfig(
    masking_strategy=MaskingStrategy(method="entropy", num_mutations=5),
    device="cuda",  # Change to "cpu" if no GPU available
)

strategy_result = run_esm3_sample(strategy_input, strategy_config)

In [13]:
# The model automatically selected the 5 highest-entropy positions to mutate
original = "MVLSPADKTNVKAAW"
for i, seq in enumerate(strategy_result.sequences):
    diffs = [j + 1 for j, (a, b) in enumerate(zip(original, seq)) if a != b]
    print(f"Original: {original}")
    print(f"Sampled:  {seq}  (mutated positions: {diffs})")

Original: MVLSPADKTNVKAAW
Sampled:  MVLKPADKTGLSAAI  (mutated positions: [4, 10, 11, 12, 15])


### `run_esm3_score`

Pseudo-perplexity scoring masks each position in a sequence one at a time and measures how well the model predicts the original residue. Because ESM3 was trained on multimodal data, its perplexity scores reflect not just evolutionary conservation but also structural and functional plausibility. Lower perplexity indicates that the sequence is more consistent with the patterns the model learned during training. This is useful for ranking designed sequences, filtering out poor candidates before experimental validation, or identifying anomalous regions within a protein.

In [14]:
from proto_tools import (
    ESM3ScoringConfig,
    ESM3ScoringInput,
    run_esm3_score,
)

In [15]:
# Display docs
display_api_reference("esm3", "input", "run_esm3_score")

# Two short sequences to score
inputs = ESM3ScoringInput(sequences=["MKTAYIAKQR", "EVQLVESGGS"])

**Input** — `MaskedModelInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Protein sequence(s) to process. Can be provided as: |

In [16]:
# Display config docs
display_api_reference("esm3", "config", "run_esm3_score")

# Process 16 masked variants per forward pass | see docs above for all fields
config = ESM3ScoringConfig(
    batch_size=16,
    return_logits=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `ESM3ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_checkpoint` | `string` | `esm3_sm_open_v1` | ESM3 model checkpoint to use. Currently available: `"esm3_sm_open_v1"` (small open-source model). |
| `batch_size` | `integer` | `1` | Number of masked sequence variants to process per forward pass. For a sequence of length L, scoring requires L forward passes (one per position). This parameter controls how many of those masked variants are batched together. Larger batches improve throughput but use more GPU memory; reduce if encountering out-of-memory errors. |
| `return_logits` | `boolean` | `False` | Whether to include per-position logits in the output. When `True`, returns logits for each sequence. When `False`, only returns metrics (saves memory and serialization time). Default: `False`. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages during scoring. |
| `device` | `string` | `cuda` | Device to run the model on. Options include `"cuda"`, `"cpu"`, `"mps"`, or specific GPU devices like `"cuda:0"`. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [17]:
# Run the scoring function
score_result = run_esm3_score(inputs, config)

Running run_esm3_score [00:00]

In [18]:
# Display output docs
display_api_reference("esm3", "output", "run_esm3_score")

# Show all scoring metrics for each input sequence
for i, score in enumerate(score_result.scores):
    print(f"Sequence {i + 1}: {inputs.sequences[i]}")
    print(f"  Log-likelihood:     {score['log_likelihood']:.3f}")
    print(f"  Avg log-likelihood: {score['avg_log_likelihood']:.3f}")
    print(f"  Perplexity:         {score['perplexity']:.3f}")

**Output** — `MaskedModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `scores` | `List[MaskedModelScoringMetrics]` | required | List of scoring outputs, one per input sequence. Each entry contains metrics (log\_likelihood, avg\_log\_likelihood, perplexity) and optional per-position logits. |
| `metrics` | `Dict[string, number]` |  | Dictionary of scalar scoring metrics. |
| `logits` | `array` |  | Optional per-position logits array. |
| `vocab` | `array` |  | Optional token ordering for logits; logits\[:, j] corresponds to vocab\[j]. |

Sequence 1: MKTAYIAKQR
  Log-likelihood:     -27.781
  Avg log-likelihood: -2.778
  Perplexity:         16.089
Sequence 2: EVQLVESGGS
  Log-likelihood:     -27.734
  Avg log-likelihood: -2.773
  Perplexity:         16.014


## Export Results

Results from each function can be exported to standard file formats for downstream analysis. Embeddings export to CSV with one row per sequence, sampled sequences export to FASTA for compatibility with other bioinformatics tools, and scores export to JSON with full metadata.

In [19]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

embeddings_result.export("esm3_embeddings", export_path=output_dir, file_format="csv")
print(f"Embeddings exported to {output_dir / 'esm3_embeddings.csv'}")

strategy_result.export("esm3_sequences", export_path=output_dir, file_format="fasta")
print(f"Sampled sequences exported to {output_dir / 'esm3_sequences.fasta'}")

score_result.export("esm3_scores", export_path=output_dir, file_format="json")
print(f"Scores exported to {output_dir / 'esm3_scores.json'}")

Embeddings exported to example_output/esm3_embeddings.csv
Sampled sequences exported to example_output/esm3_sequences.fasta
Scores exported to example_output/esm3_scores.json
